# Pronóstico horario de PM2.5 en Lima Metropolitana

**1ACC0057 Machine Learning · UPC**

El CSV original de SENAMHI usa **UTC** y el modelo trabaja en **America/Lima (UTC−5)**. La comprobación con OpenAQ encontró 112 coincidencias en 112 pares de cuatro estaciones dentro del redondeo de publicación. La unidad de PM2.5 es **µg/m³**.

El cuaderno presenta la comparación inicial a una hora y el pronóstico directo a 1, 3 y 6 horas. Con `RECALCULAR = False` se abren los resultados ya verificados. Para volver a entrenar, instala las dependencias de `requirements.txt` y apunta `DATOS` a los insumos originales indicados en el README.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

COLAB = "google.colab" in sys.modules or Path("/content").is_dir()
PREDETERMINADO = Path("/content/Viento-Lima") if COLAB else Path.cwd()
PROYECTO = Path(os.environ.get("PM25_PROYECTO", PREDETERMINADO)).resolve()
if COLAB and not PROYECTO.exists():
    subprocess.run(["git", "clone", "--branch", "develop", "--single-branch",
                    "https://github.com/KeniChris/Viento-Lima.git", str(PROYECTO)], check=True)
if not (PROYECTO / "codigo" / "preparar.py").is_file():
    raise FileNotFoundError("Fija PM25_PROYECTO a la carpeta que contiene codigo/")
sys.path.insert(0, str(PROYECTO))
DATOS = Path(os.environ.get("PM25_DATOS", PROYECTO / "datos"))
RESULTADOS = PROYECTO / "resultados"
RECALCULAR = False
RECALCULAR_HORIZONTES = False
EJECUTAR_HITO2 = False
HILOS = 4

## Verificación y preparación de los datos

OpenAQ publica fecha, hora, desfase y unidad de lecturas del SENAMHI. Por ejemplo, en San Borja el 15 de abril de 2024 se registraron **10,19 µg/m³** a las **06:00 UTC**, equivalentes a **01:00 en Lima**. La segunda interpretación se usa en la tabla como control de la verificación.

El pipeline conserva la marca UTC, convierte a Lima y cruza por estación y hora con la meteorología. Se preservan los faltantes de PM2.5 y se incorporan velocidad y dirección del viento IFS a 10 m. Al aplicar la convención meteorológica de procedencia, un viento del norte se representa como movimiento hacia el sur.

In [ ]:
if RECALCULAR:
    from codigo.actualizar_aire import verify_clock
    from codigo.preparar import prepare_dataset
    from codigo.viento import WIND_FILE, build_wind_file
    VIENTO = DATOS / WIND_FILE
    if not VIENTO.is_file():
        build_wind_file(VIENTO)
    verify_clock(DATOS / "aire_lima_completo.csv", RESULTADOS)
    matrix = prepare_dataset(DATOS, VIENTO, RESULTADOS)
display(pd.read_csv(RESULTADOS / "verificacion_horaria.csv"))
display(pd.read_csv(RESULTADOS / "ejemplos_hora_verificada.csv").head(8))
display(pd.read_csv(RESULTADOS / "auditoria_preparacion.csv"))

## Comparación inicial a una hora con horas corregidas

Los pliegues expansivos validan en 2020, 2021 y 2022 y descartan las etiquetas que cruzan cada corte. La prueba histórica usa 56.035 pronósticos de 2023 a abril de 2024. La ablación mantiene los hiperparámetros de XGBoost y añade sucesivamente viento local, PM2.5 vecino y predictores de transporte.

Random Forest fue seleccionado por MAE medio entre estaciones en validación. En la prueba histórica, XGBoost base obtuvo el menor MAE y Random Forest el menor RMSE. Añadir transporte sobre el XGBoost con lecturas vecinas aumentó el MAE de 2,910 a 2,925 µg/m³.

In [ ]:
if RECALCULAR:
    from codigo.evaluar import evaluate_models
    evaluate_models(matrix, RESULTADOS, HILOS)
cv = pd.read_csv(RESULTADOS / "cv_modelos.csv")
test = pd.read_csv(RESULTADOS / "prueba_modelos.csv")
display(cv[["modelo", "MAE_macro_cv", "DE_MAE_macro_cv", "RMSE_cv"]].round(4))
display(test[["modelo", "n", "MAE", "MAE_macro", "RMSE", "R2",
              "mae_altos", "sesgo_altos", "sensibilidad_altos"]].round(4))
display(pd.read_csv(RESULTADOS / "diferencias_prueba.csv").round(4))
display(pd.read_csv(RESULTADOS / "prueba_por_estacion.csv")
        [["modelo", "ESTACION", "n", "MAE", "mae_altos"]].round(4))

## Prueba posterior al corte de SENAMHI

Las lecturas de OpenAQ posteriores al 30 de abril de 2024 se evalúan con los modelos ajustados con datos anteriores a 2023. La cobertura y los códigos de respuesta del archivo diario documentan los intervalos disponibles.

In [ ]:
if RECALCULAR:
    from codigo.actualizar_aire import EXTENSION_FILE, download_extension
    from codigo.validar_extension import evaluate_extension
    if not (DATOS / EXTENSION_FILE).is_file():
        download_extension(DATOS, RESULTADOS)
    evaluate_extension(DATOS, RESULTADOS, jobs=HILOS)
display(pd.read_csv(RESULTADOS / "cobertura_openaq_2024.csv"))
display(pd.read_csv(RESULTADOS / "acceso_openaq_2024.csv"))
display(pd.read_csv(RESULTADOS / "prueba_extension_2024.csv").round(4))
display(pd.read_csv(RESULTADOS / "diferencia_transporte_extension_2024.csv").round(4))

## Pronóstico directo a 1, 3 y 6 horas

Se ajusta un modelo por horizonte con objetivo `PM2.5(t+h) − PM2.5(t)`. El pronóstico suma el cambio estimado a la última concentración. Las etiquetas se buscan por hora exacta, sin rellenar mediciones ausentes. La cohorte empieza en 2017 para cubrir el archivo IFS y exige PM2.5 observado en la emisión y en la hora objetivo. Los modelos de un mismo horizonte se comparan sobre las mismas filas.

La comparación incremental añade a la historia local la velocidad y los componentes del viento, las observaciones vecinas, el transporte entre estaciones y dos grupos de focos térmicos. La dirección meteorológica indica la procedencia. La proyección del vector de desplazamiento sobre el rumbo origen–receptor estima qué porción de la velocidad empuja hacia la estación. Una proyección de 3 m/s equivale a 10,8 km/h. Los predictores ponderan el PM2.5 y el incremento de las tres horas anteriores cuando la distancia recorrida es compatible con la hora objetivo.

Se filtraron 3 738 detecciones VIIRS S-NPP de fuego probable con confianza nominal o alta. Se comparó el conteo y la potencia radiativa de detecciones dentro de 50 km con un ponderador adicional por rumbo y tiempo de viaje. El proxy usa viento histórico de la celda de la estación más cercana al foco en la hora de adquisición y un retraso simulado de tres horas. FRP mide potencia radiativa en MW. El archivo histórico permite estudiar asociación predictiva y depende del paso del satélite. [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/content/descriptions/FIRMS_VIIRS_Firehotspots.html)

In [ ]:
RUTA_HORIZONTES = RESULTADOS / "multihorizonte"

if RECALCULAR_HORIZONTES:
    from codigo.actualizar_aire import EXTENSION_FILE
    from codigo.pronosticar import HORIZONS, run
    from codigo.viento import WIND_FILE, build_wind_file

    viento = DATOS / WIND_FILE
    if not viento.is_file():
        build_wind_file(viento)
    if not (DATOS / EXTENSION_FILE).is_file():
        raise FileNotFoundError("Ejecuta python -m codigo.actualizar_aire --extender para obtener las lecturas adicionales")
    run(DATOS, viento, RUTA_HORIZONTES, HORIZONS, HILOS, True, None, include_fires=True)

validacion_h = pd.read_csv(RUTA_HORIZONTES / "validacion.csv")
prueba_h = pd.read_csv(RUTA_HORIZONTES / "prueba.csv")
ablacion_h = pd.read_csv(RUTA_HORIZONTES / "ablacion.csv")

### Validación y ajuste

Como en los laboratorios de regresión y ensambles del curso, se comparan los modelos en las mismas particiones, se elige por validación y se revisan los residuos de prueba. En esta serie se avanza cronológicamente con validación en 2020, 2021 y 2022. La mediana, los indicadores de ausencia y el escalado de Ridge se ajustan en cada entrenamiento. Las etiquetas que cruzan un corte quedan fuera del entrenamiento.

Se prueban Ridge con α 1 y 100, HistGradientBoosting con 15 y 31 hojas y XGBoost con profundidades 3 y 5 sobre los predictores de vecinos. El ensamble promedia XGBoost de profundidad 3 e HistGradientBoosting de 15 hojas con pesos iguales. En la comparación incremental se fija XGBoost de profundidad 3 y se modifica únicamente el conjunto de entradas. Se selecciona el menor MAE medio por estación, promediado entre los tres pliegues. Los modelos se reajustan con objetivos anteriores a 2023.

[Laboratorio de regresión](https://drive.google.com/file/d/1IXLm6uqucrX56UXb0b0mCgY9KaVTAlGt/view) · [Laboratorio de ensambles](https://drive.google.com/file/d/12VMzOfKEfdWcS91ICLUAPduDP-2VHaEt/view)

In [ ]:
for horizonte, resultados in validacion_h.groupby("horizonte_h"):
    print(f"Horizonte de {horizonte} h")
    display(resultados[["modelo", "familia", "MAE_macro_cv", "DE_MAE_macro_cv", "RMSE_cv", "seleccionado"]].round(4))

### Evaluación histórica y cruces nuevos

El MAE mide el error típico en µg/m³. El RMSE da más peso a los errores grandes. R² compara el error cuadrático con el de usar la media de la prueba. La prueba histórica reutiliza enero de 2023 a abril de 2024 para comparar con el análisis anterior. La extensión de OpenAQ cubre cuatro estaciones y permite revisar el comportamiento en un periodo posterior.

Un cruce nuevo ocurre si la última medición está por debajo del percentil 95 de su estación en desarrollo y la medición objetivo lo supera o iguala. Cada horizonte y estación tiene su umbral calculado antes de la prueba. La persistencia tiene cero aciertos de cruces por definición. Es un diagnóstico técnico de anticipación. El INCA peruano para PM2.5 se calcula sobre un promedio móvil de 24 horas. [MINAM](https://sinia.minam.gob.pe/sites/default/files/sinia/archivos/public/docs/rm-n-181-2016-minam.pdf)

| Horizonte | Modelo elegido en validación | MAE | MAE persistencia | Cruces anticipados |
| --- | --- | ---: | ---: | ---: |
| 1 h | Promedio XGBoost + HistGradientBoosting | 2,823 | 3,193 | 10/300 |
| 3 h | XGBoost con viento local | 4,938 | 6,362 | 0/522 |
| 6 h | XGBoost local | 5,762 | 8,529 | 0/623 |

In [ ]:
elegidos_h = validacion_h.loc[validacion_h.seleccionado, ["horizonte_h", "modelo"]]
evaluacion_h = prueba_h.loc[prueba_h.subconjunto.eq("todas")].merge(elegidos_h, on=["horizonte_h", "modelo"])
columnas = ["periodo", "horizonte_h", "modelo", "n", "MAE", "RMSE", "R2", "recall_altos",
            "aciertos_cruce", "n_cruces", "precision_cruces"]
display(evaluacion_h[columnas].sort_values(["periodo", "horizonte_h"]).round(4))
display(pd.read_csv(RUTA_HORIZONTES / "por_estacion.csv").query('periodo == "historica"')
        [["horizonte_h", "ESTACION", "modelo", "n", "MAE", "RMSE", "recall_cruces"]].round(3))

### Residuos

El residuo es la concentración observada menos el pronóstico. Un valor positivo indica subestimación. Se muestran la mediana y los percentiles 10–90 de los residuos para rangos de pronóstico con al menos 30 casos. Las bandas describen errores observados en la prueba y su dispersión, sin representar intervalos de pronóstico para una emisión nueva.

In [ ]:
import matplotlib.pyplot as plt

diagnostico = pd.read_csv(RUTA_HORIZONTES / "residuos.csv")
historico = diagnostico.loc[diagnostico.periodo.eq("historica") & diagnostico.n.ge(30)]
fig, axes = plt.subplots(1, 3, sharey=True, figsize=(10.4, 3.4), layout="constrained")
for horizonte, ax in zip((1, 3, 6), axes):
    datos = historico.loc[historico.horizonte_h.eq(horizonte)]
    x = datos.pronosticado_mediana.to_numpy()
    ax.fill_between(x, datos.residuo_p10, datos.residuo_p90,
                    color="#91BDB9", alpha=.55, label="P10–P90")
    ax.plot(x, datos.residuo_mediana, color="#185D59", marker="o", label="Mediana")
    ax.axhline(0, color="#52636B", linewidth=1, linestyle="--")
    ax.set_title(f"{horizonte} h de anticipación", loc="left", fontweight="semibold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=.22)
axes[0].set_ylabel("Observado − pronosticado (µg/m³)")
axes[-1].legend(loc="upper left", frameon=False)
fig.supxlabel("PM2.5 pronosticado (µg/m³)")
plt.show()


### ¿Qué aportaron el viento y los focos?

Cada fila compara pronósticos de XGBoost sobre las mismas estaciones y horas. Un ΔMAE negativo favorece el grupo que añade la variable. Los intervalos descriptivos usan remuestreo en bloques semanales. `alza_origen_alineada` conserva los casos cuyo aumento máximo de tres horas en un origen alineado alcanza el percentil 90 de los aumentos positivos de desarrollo. `con_focos_alineados` indica señal térmica ponderada positiva. El tamaño de estos subconjuntos y el número de semanas determinan la estabilidad de la comparación.

Para un pronóstico operativo, se necesita registrar cuándo están disponibles las lecturas, las detecciones y cada corrida meteorológica. El experimento retrospectivo usa meteorología de archivo con marca horaria hasta la emisión. Probar una trayectoria con viento cambiante requiere pronósticos meteorológicos emitidos antes de esa hora. [Archivo de corridas de Open-Meteo](https://open-meteo.com/en/docs/single-runs-api)

In [ ]:
comparacion_h = ablacion_h.loc[ablacion_h.modelo.isin(["xgb_transporte", "xgb_focos_cercanos", "xgb_focos_alineados"])]
display(comparacion_h.sort_values(["periodo", "horizonte_h", "subconjunto", "modelo"]).round(4))

### Evaluación por episodios después de un alza en origen

La evaluación de alertas se activa cuando la subida de PM2.5 de una estación en tres horas alcanza su percentil 95 y la concentración alcanza su percentil 75. Se observan estaciones a 20 km o menos con concentración inicial bajo su percentil 75. El objetivo es una primera alza anómala en las seis horas siguientes. Los percentiles por estación se fijaron con datos de 2017 a 2022. Se califican únicamente ventanas completas.

El rumbo se considera compatible si el viento de la hora de emisión va a menos de 45° del rumbo origen–receptora, o en el límite, y su velocidad proyectada alcanza para recorrer la distancia en seis horas. Los demás pares alcanzables sirven de comparación. Ambos clasificadores usan las mismas observaciones, distancia, hora y velocidad. El segundo añade el rumbo y el alcance proyectado. Los dos se comparan con el mismo cupo retrospectivo de alertas.

| Periodo | Alzas con viento compatible | Alzas en otros pares alcanzables | Aciertos sin rumbo | Aciertos con rumbo |
| --- | ---: | ---: | ---: | ---: |
| Enero de 2023 a abril de 2024 | 36/257 (14,0 %) | 64/1.019 (6,3 %) | 46/257 | 48/257 |
| Mayo y junio de 2024 | 4/32 (12,5 %) | 8/44 (18,2 %) | 6/32 | 6/32 |

La dirección se asoció con más alzas en el periodo histórico. Como señal adicional del clasificador, aportó dos aciertos con el mismo cupo. La regla física sola generó 221 falsas alertas en 257 avisos. En la comprobación posterior de cuatro estaciones, los dos clasificadores acertaron la misma cantidad.

Para reproducir la comparación histórica ejecuta `python -m codigo.eventos_viento`. Añade `--extender` si ya descargaste las lecturas posteriores de OpenAQ.

In [ ]:
ruta_eventos = RESULTADOS / "eventos_viento"
if ruta_eventos.is_dir():
    for tabla in ("cobertura", "direccion", "alertas", "diferencias"):
        display(pd.read_csv(ruta_eventos / f"{tabla}.csv").round(4))
else:
    print("Actualiza el repositorio a develop para consultar las tablas por episodio.")

### Diagnóstico del MAE y matriz de confusión

Los modelos de concentración se eligen por MAE medio por estación en los pliegues cronológicos de 2020, 2021 y 2022. Se añadieron rezagos, antecedentes de la misma hora en días anteriores, altura de capa límite y ventilación. Se compararon XGBoost con pérdidas cuadrática y absoluta, HistGradientBoosting y ExtraTrees. La referencia anterior se reprodujo en las mismas filas.

| Horizonte | Histórico previo | Histórico XGBoost con rezagos y pérdida absoluta | OpenAQ previo | OpenAQ del candidato |
| --- | ---: | ---: | ---: | ---: |
| 1 h | 2,823 | 2,770 | 4,556 | 4,613 |
| 3 h | 4,938 | 4,706 | 8,116 | 8,473 |
| 6 h | 5,762 | 5,353 | 9,851 | 10,363 |

En las mismas cuatro estaciones, el porcentaje de casos altos a una hora pasó de 1,6 % a 21,8 % entre los periodos. La altura de capa límite tuvo un efecto pequeño cuando estuvo disponible. El archivo de altura utilizado termina en 2023. En horas sin altura se aplica el modelo equivalente entrenado sin esas variables.

Para las alertas, los percentiles y modelos se ajustaron con datos anteriores a 2022. En 2022 se fijó cada umbral para detectar los mismos 83 casos que la regla física. La matriz histórica contiene 1 214 pares con ventanas completas.

| Método | VP | FP | FN | VN | Precisión | Sensibilidad |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Regla de viento | 25 | 191 | 67 | 931 | 11,6 % | 27,2 % |
| Logístico sin rumbo | 22 | 87 | 70 | 1 035 | 20,2 % | 23,9 % |
| Logístico con rumbo | 22 | 83 | 70 | 1 039 | 21,0 % | 23,9 % |

VP cuenta alzas detectadas, FP falsas alertas, FN alzas omitidas y VN ausencias de alza sin aviso. En OpenAQ ambos clasificadores registraron 3 aciertos y 11 falsas alertas en 68 pares.

Para reproducir el diagnóstico, ejecuta `python -m codigo.analizar_mae` y `python -m codigo.eventos_viento --calibrar`. Añade `--extender` si ya descargaste los datos posteriores de OpenAQ.

[LSTM en el estudio de Montalvo et al. sobre Lima](https://doi.org/10.3389/frsc.2022.849762) · [PM2.5 de CAMS](https://ads.atmosphere.copernicus.eu/datasets/cams-global-atmospheric-composition-forecasts?tab=overview)

In [ ]:
error_dir = RESULTADOS / "analisis_mae"
matrices = RESULTADOS / "eventos_viento" / "alertas_umbral_validado.csv"
if (error_dir / "prueba.csv").is_file() and matrices.is_file():
    error = pd.read_csv(error_dir / "prueba.csv")
    nombres = ["referencia_previa", "xgb_historia_mae"]
    columnas = ["periodo", "horizonte_h", "modelo", "n", "MAE", "RMSE", "R2", "n_altos", "MAE_altos"]
    display(error.loc[error.subconjunto.eq("todas") & error.modelo.isin(nombres), columnas].round(4))
    display(pd.read_csv(matrices).round(4))
else:
    print("Actualiza el repositorio a develop para consultar las métricas y matrices.")

## Integración con PostgreSQL

La prueba local leyó las siete entradas de una hora, escribió siete pronósticos para la hora siguiente y comprobó la repetición sin duplicados. El contrato SQL y el adaptador están en `codigo/`. La instancia común del módulo de IA requiere una comprobación independiente.

La evidencia usa filas históricas y el modelo de una hora. El servicio para varios horizontes requiere publicar las entradas a tiempo y validar la recepción desde la base común del módulo de IA.

In [ ]:
print((RESULTADOS / "conexion_postgresql.json").read_text(encoding="utf-8"))
display(pd.read_csv(RESULTADOS / "conexion_pronosticos.csv").round(4))

## Fuentes

- [SENAMHI · Monitoreo de contaminantes de Lima Metropolitana](https://www.datosabiertos.gob.pe/dataset/monitoreo-de-los-contaminantes-del-aire-en-lima-metropolitana-servicio-nacional-de)
- [OpenAQ · Formato de datos y desfase horario](https://docs.openaq.org/aws/about)
- [OpenAQ · San Borja](https://explore.openaq.org/locations/2402)
- [Open-Meteo · Viento histórico de ECMWF IFS](https://open-meteo.com/en/docs/historical-weather-api)
- [Viento-Lima · Rama develop](https://github.com/KeniChris/Viento-Lima/tree/develop)